In [8]:
from pathlib import Path
import hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

# Usa la misma ruta configurada en el notebook anterior
DATASET_DIR = Path("/kaggle/input/datasets/claudiobances/coffee-leaf-dataset-by-phytosanitary-class/coffee_leaves/")

CLASSES = ["Roya", "Sanas"]
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9
assert DATASET_DIR.exists(), f"No existe la ruta: {DATASET_DIR}"

# Inventario de imágenes
records = []

for class_name in CLASSES:
    class_dir = DATASET_DIR / class_name

    if not class_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta: {class_dir}")

    for path in sorted(class_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in EXTENSIONS:
            records.append({
                "ruta": str(path.resolve()),
                "clase": class_name,
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
            })

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No se encontraron imágenes.")

# Comprobar que no haya hashes repetidos en el inventario
if df["sha256"].duplicated().any():
    raise ValueError(
        "Se detectaron imágenes con hashes repetidos. "
        "Revisa el inventario antes de continuar."
    )

# Separar entrenamiento del resto, estratificando por clase
train_df, remaining_df = train_test_split(
    df,
    test_size=VAL_RATIO + TEST_RATIO,
    random_state=SEED,
    stratify=df["clase"]
)

# Dividir el resto por igual entre validación y prueba
val_df, test_df = train_test_split(
    remaining_df,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=remaining_df["clase"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

manifest = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Guardar el manifiesto en ml/reports
REPORTS_DIR = Path("/kaggle/working/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = REPORTS_DIR / "dataset_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Total de imágenes:", len(manifest))
print("\nDistribución por conjunto y clase:")
print(pd.crosstab(manifest["split"], manifest["clase"], margins=True))
print("\nManifiesto guardado en:", manifest_path.resolve())
print("\nSemilla utilizada:", SEED)

Total de imágenes: 1000

Distribución por conjunto y clase:
clase  Roya  Sanas   All
split                   
test     75     75   150
train   350    350   700
val      75     75   150
All     500    500  1000

Manifiesto guardado en: /kaggle/working/reports/dataset_manifest.csv

Semilla utilizada: 42


In [10]:
from pathlib import Path
import pandas as pd

manifest_path = Path("/kaggle/working/reports/dataset_manifest.csv")
manifest = pd.read_csv(manifest_path)

# 1. Comprobar que las columnas necesarias existan
required_columns = {"ruta", "clase", "sha256", "split"}
assert required_columns.issubset(manifest.columns), (
    f"Faltan columnas: {required_columns - set(manifest.columns)}"
)

# 2. Comprobar que no haya rutas duplicadas
assert not manifest["ruta"].duplicated().any(), (
    "Hay rutas duplicadas en el manifiesto."
)

# 3. Comprobar que no haya hashes repetidos
assert not manifest["sha256"].duplicated().any(), (
    "Hay hashes repetidos en el manifiesto."
)

# 4. Comprobar las cantidades esperadas
expected = {
    ("train", "Roya"): 350,
    ("train", "Sanas"): 350,
    ("val", "Roya"): 75,
    ("val", "Sanas"): 75,
    ("test", "Roya"): 75,
    ("test", "Sanas"): 75,
}

actual = manifest.groupby(["split", "clase"]).size().to_dict()

for key, expected_count in expected.items():
    assert actual.get(key, 0) == expected_count, (
        f"{key}: esperado {expected_count}, "
        f"obtenido {actual.get(key, 0)}"
    )

# 5. Comprobar que los conjuntos sean los previstos
assert set(manifest["split"]) == {"train", "val", "test"}
assert set(manifest["clase"]) == {"Roya", "Sanas"}

print("VERIFICACIÓN COMPLETADA CORRECTAMENTE")
print("Total de imágenes:", len(manifest))
print("No hay rutas duplicadas.")
print("No hay hashes duplicados.")
print("Las cantidades por conjunto y clase son correctas.")

VERIFICACIÓN COMPLETADA CORRECTAMENTE
Total de imágenes: 1000
No hay rutas duplicadas.
No hay hashes duplicados.
Las cantidades por conjunto y clase son correctas.


In [11]:
from pathlib import Path
import pandas as pd
import json

# Cargar el manifiesto
manifest_path = Path("/kaggle/working/reports/dataset_manifest.csv")
manifest = pd.read_csv(manifest_path)

# Calcular distribución por conjunto y clase
distribution = (
    manifest
    .groupby(["split", "clase"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

# Añadir totales
distribution["Total"] = distribution.sum(axis=1)
distribution.loc["Total"] = distribution.sum(axis=0)

print("DISTRIBUCIÓN DEL DATASET")
print(distribution)

# Crear carpeta de reportes
reports_dir = Path("/kaggle/working/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# Guardar reporte CSV
distribution.to_csv(reports_dir / "distribucion_dataset.csv")

# Guardar configuración reproducible
config = {
    "dataset": "Coffee leaves - Saposoa",
    "clases_incluidas": ["Roya", "Sanas"],
    "clases_excluidas": ["Ojo_Gallo"],
    "total_imagenes": int(len(manifest)),
    "proporcion": {
        "train": 0.70,
        "val": 0.15,
        "test": 0.15
    },
    "random_seed": 42,
    "augmentation": {
        "aplicada": False,
        "nota": "No se ha aplicado aumento de datos."
    }
}

with open(
    reports_dir / "configuracion_split.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("\nArchivos generados:")
print("- reports/distribucion_dataset.csv")
print("- reports/configuracion_split.json")

DISTRIBUCIÓN DEL DATASET
clase  Roya  Sanas  Total
split                    
train   350    350    700
val      75     75    150
test     75     75    150
Total   500    500   1000

Archivos generados:
- reports/distribucion_dataset.csv
- reports/configuracion_split.json
